# Module 3: Hosting an MCP Server on Amazon Bedrock AgentCore Runtime

## Overview
In this module, we will learn how to **host MCP (Model Context Protocol) servers** on the **Amazon Bedrock AgentCore Runtime**.  
We will use the **Amazon Bedrock AgentCore Python SDK** to wrap our MCP tools into a server that communicates using the standardized AgentCore MCP protocol.

The **AgentCore SDK** handles the low-level server implementation, allowing you to focus on the **tool logic** rather than protocol plumbing.

---

## Module Details

| **Information**     | **Details**                                                  |
|--------------------|--------------------------------------------------------------|
| Module type        | Hosting Tools                                                |
| Tool type          | MCP Server                                                   |
| Module components  | Hosting MCP server on AgentCore Runtime                      |
| Module vertical    | Cross-vertical                                               |
| SDK used           | Amazon BedrockAgentCore Python SDK + MCP library             |

---

## Module Architecture

In this module, we will **deploy an MCP server** to the AgentCore Runtime.  
The MCP server will include **five custom tools** that interact with both **structured data** (Amazon DynamoDB) and **unstructured data** (Amazon Bedrock Knowledge Base).

<div style="text-align:left">
    <img src="images/invoke.png" width="85%"/>
</div>

---

## Key Capabilities Covered

- Creating MCP servers with **custom tool endpoints**
- **Testing** MCP servers locally
- **Deploying** MCP servers to Amazon Bedrock AgentCore Runtime
- **Invoking** the deployed MCP server with secure authentication

---

## Jupyter Notebook Introduction

Jupyter Notebooks are an interactive environment where you can write and execute code in small chunks called "cells." As you go through the workshop, you will execute code cells in the order they appear. There are several ways to run a cell. Once you click into a cell, the outside border will be highlighted. Then you can click the play button on the top tool bar to run that specific cell. Another way to run the cell is to *press Shift + Return* to execute the code. Before you run a cell, you will see an open square bracket to the left of the cell, [ ]. While the cell is executing, you will see an asterisk inside the bracket, [*]. Once the cell is executed, you will see a number in the bracket, [1]. The number in the bracket next to the cell indicates the execution order of that cell within the notebook.

So before a cell is run, you will see something like the following:

<div style="text-align:left">
    <img src="images/prerun.png" width="85%"/>
</div>

Then, you can run the cell and while it is executing you will see something similar to the following:

<div style="text-align:left">
    <img src="images/executing.png" width="85%"/>
</div>

Once the cell is run, it will print any output under the cell and showcase the execution order in the brackets like the following:

<div style="text-align:left">
    <img src="images/ran.png" width="85%"/>
</div>

---

## Select the Correct Kernel

A kernel in Jupyter Notebooks is the computational engine that executes the code you write in your notebook. Think of it as the "brain" that runs your code and returns the results. Each kernel is tied to a specific programming language, such as Python, or R, and maintains the state of your variables and functions as you work through your notebook.

Before running commands, ensure you are using the correct kernel:

1. Select the **Kernel** dropdown at the top-right of the notebook.

<div style="text-align:left">
    <img src="images/click_kernel.png" width="85%"/>
</div>

2. Choose **Python 3 (ipykernel)**.
3. Click **Select** to confirm.

<div style="text-align:left; margin-top: 10px;">
    <img src="images/choose_kernel.png" width="50%"/>
</div>

---

## Next Step
We will begin by installing the required dependencies. Then we will **initiate a boto3 session** and confirm the execution role running in SageMaker Studio. **The pip install may have pip dependency resolver warnings, that is to be expected and you can continue with the workshop. You can ignore the warnings and continue the workshop.**


In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
import boto3
# This will automatically use ~/.aws/credentials
session = boto3.Session()


In [ ]:
!aws sts get-caller-identity

## Understanding MCP (Model Context Protocol)

**MCP** is a protocol that enables AI agents to securely access external **data**, **APIs**, and **tools** in a structured and secure way.

### Key Concepts

| Concept | Meaning |
|--------|---------|
| **Tools** | Functions the agent can call to perform real actions |
| **Streamable HTTP** | Transport protocol used by AgentCore Runtime for tool invocation |
| **Session Isolation** | Each client session is isolated via the `Mcp-Session-Id` header |
| **Stateless Operation** | MCP servers must be stateless to scale across concurrent sessions |

> **AgentCore Runtime Requirement:**  
> MCP servers must listen on:  
> **0.0.0.0:8000/mcp**

---

## Project Structure

We will use the following folder layout:

    agentcore_resources/module_3
    ├── mcp_server.py              # Main MCP server implementation
    ├── my_mcp_client.py           # Local testing client
    ├── my_mcp_client_remote.py    # Remote testing client
    └── requirements.txt           # Python dependencies

---

## Creating the MCP Server

We will create an MCP server using **FastMCP** with `stateless_http=True`, which is required for compatibility with the **AgentCore Runtime**, ensuring each request is independently handled.

<div style="text-align:left; margin-top: 10px;">
    <img src="images/mcp.png" width="50%"/>
</div>

---

## MCP Server Tools Overview

The MCP server integrates with:

- **Amazon DynamoDB** (structured product and transaction data)
- **Amazon Bedrock Knowledge Base** (semantic, vector-based product search)

### Available Tools

| Tool Name | Purpose |
|----------|---------|
| `search_product_knowledge(query: str)` | Performs semantic product lookup in the Bedrock Knowledge Base and returns the top matches. |
| `list_products()` | Retrieves all products from DynamoDB. |
| `get_product(product_id: str)` | Fetches detailed product metadata for a given product ID. |
| `check_inventory(product_id: str)` | Returns current stock levels and availability. |
| `place_order(product_id: str, quantity: int)` | Simulates placing an order, updates inventory, and records the transaction. |

---

## ⚠️ IMPORTANT – Manual Step Required

Before running the next cell:

### Copy the **Knowledge Base ID** from **Module 2** and assign it to the variable shown in the upcoming configuration cell.

Note that the **Knowledge Base ID** is not the same as the **Knowledge Base Name**.

If this step is skipped:

- The `search_product_knowledge` tool will **fail**
- The chatbot will **not** be able to provide product recommendations or descriptions

> This step ensures the MCP server is correctly linked to your Knowledge Base.


In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP
from starlette.responses import JSONResponse
import boto3
import json
from typing import Dict, Any
from decimal import Decimal

mcp = FastMCP(host="0.0.0.0", stateless_http=True)
table_name = 'secure-ecommerce-stack-product-catalog' #this is the product table name
transaction_name = 'secure-ecommerce-stack-transaction-logs' #this is the transaction table to store product orders
knowledge_base_id = '' #ENTER KB ID

@mcp.tool()
def search_product_knowledge(query: str) -> str:
    """Search product information from Bedrock knowledge base"""
    try:
        bedrock_agent = boto3.client('bedrock-agent-runtime')
        
        response = bedrock_agent.retrieve(
            knowledgeBaseId=knowledge_base_id,
            retrievalQuery={
                'text': query
            },
            retrievalConfiguration={
                'vectorSearchConfiguration': {
                    'numberOfResults': 5
                }
            }
        )
        
        results = []
        for result in response.get('retrievalResults', []):
            content = result.get('content', {}).get('text', '')
            score = result.get('score', 0)
            results.append({
                'content': content,
                'confidence_score': score
            })
        
        if results:
            return json.dumps({
                'status': 'success',
                'query': query,
                'results': results
            }, indent=2)
        else:
            return json.dumps({
                'status': 'no_results',
                'query': query,
                'message': 'No product information found for the given query'
            })
            
    except Exception as e:
        return json.dumps({
            'status': 'error',
            'query': query,
            'error': str(e)
        })

@mcp.tool()
def list_products() -> str:
    """List all products with basic information from the product catalog"""
    
    try:
        dynamodb = boto3.resource('dynamodb')
        table = dynamodb.Table(table_name)
        
        # List all products with basic info
        response = table.scan()
        products = response.get('Items', [])
        
        if not products:
            return json.dumps({
                'status': 'no_products',
                'message': 'No products found in the database'
            })
        
        # Format product list for easy reading
        product_list = []
        for product in products:
            product_list.append({
                'product_id': product.get('product_id'),
                'name': product.get('name'),
                'price': float(product.get('price', 0)),
                'inventory_count': int(product.get('inventory_count', 0)),
                'category': product.get('category', 'Unknown')
            })
        
        return json.dumps({
            'status': 'success',
            'total_products': len(product_list),
            'products': product_list
        }, indent=2)
        
    except Exception as e:
        return json.dumps({
            'status': 'error',
            'error': str(e)
        })

@mcp.tool()
def get_product(product_id: str) -> str:
    """Get complete product information for a specific product
    
    Args:
        product_id: The product identifier
    """
    
    try:
        dynamodb = boto3.resource('dynamodb')
        table = dynamodb.Table(table_name)
        
        if not product_id:
            return json.dumps({
                'status': 'error',
                'error': 'product_id is required'
            })
        
        # Get the specific product
        response = table.get_item(Key={'product_id': product_id})
        
        if 'Item' not in response:
            return json.dumps({
                'status': 'product_not_found',
                'product_id': product_id,
                'message': f'Product {product_id} not found in database'
            })
        
        product = response['Item']
        current_inventory = int(product.get('inventory_count', 0))
        
        # Return complete product information
        return json.dumps({
            'status': 'success',
            'product': {
                'product_id': product.get('product_id'),
                'name': product.get('name'),
                'description': product.get('description'),
                'price': float(product.get('price', 0)),
                'inventory_count': current_inventory,
                'category': product.get('category'),
                'image_url': product.get('image_url'),
                'created_at': product.get('created_at'),
                'updated_at': product.get('updated_at')
            }
        }, indent=2)
        
    except Exception as e:
        return json.dumps({
            'status': 'error',
            'product_id': product_id,
            'error': str(e)
        })

@mcp.tool()
def check_inventory(product_id: str) -> str:
    """Check inventory status for a specific product
    
    Args:
        product_id: The product identifier
    """
    
    try:
        dynamodb = boto3.resource('dynamodb')
        table = dynamodb.Table(table_name)
        
        if not product_id:
            return json.dumps({
                'status': 'error',
                'error': 'product_id is required'
            })
        
        # Get the specific product
        response = table.get_item(Key={'product_id': product_id})
        
        if 'Item' not in response:
            return json.dumps({
                'status': 'product_not_found',
                'product_id': product_id,
                'message': f'Product {product_id} not found in database'
            })
        
        product = response['Item']
        current_inventory = int(product.get('inventory_count', 0))
        
        # Return inventory-focused information
        return json.dumps({
            'status': 'success',
            'product_id': product_id,
            'product_name': product.get('name'),
            'inventory_count': current_inventory,
            'price': float(product.get('price', 0)),
            'availability': 'in_stock' if current_inventory > 0 else 'out_of_stock'
        }, indent=2)
        
    except Exception as e:
        return json.dumps({
            'status': 'error',
            'product_id': product_id,
            'error': str(e)
        })

@mcp.tool()
def place_order(product_id: str, quantity: int) -> str:
    """Place an order for a specific product
    
    Args:
        product_id: The product identifier
        quantity: Quantity to order
    """
    
    try:
        from datetime import datetime
        import uuid
        from decimal import Decimal
        
        dynamodb = boto3.resource('dynamodb')
        table = dynamodb.Table(table_name)
        
        if not product_id:
            return json.dumps({
                'status': 'error',
                'error': 'product_id is required'
            })
        
        if quantity <= 0:
            return json.dumps({
                'status': 'error',
                'product_id': product_id,
                'error': 'Quantity must be greater than 0 for placing orders'
            })
        
        # Get the specific product
        response = table.get_item(Key={'product_id': product_id})
        
        if 'Item' not in response:
            return json.dumps({
                'status': 'product_not_found',
                'product_id': product_id,
                'message': f'Product {product_id} not found in database'
            })
        
        product = response['Item']
        current_inventory = int(product.get('inventory_count', 0))
        
        # Check if enough inventory is available
        if current_inventory < quantity:
            return json.dumps({
                'status': 'insufficient_inventory',
                'product_id': product_id,
                'product_name': product.get('name'),
                'requested_quantity': quantity,
                'available_quantity': current_inventory,
                'message': f'Only {current_inventory} units available, but {quantity} requested'
            })
        
        # Update inventory (reduce by ordered quantity)
        new_inventory = current_inventory - quantity
        table.update_item(
            Key={'product_id': product_id},
            UpdateExpression='SET inventory_count = :new_count, updated_at = :timestamp',
            ExpressionAttributeValues={
                ':new_count': new_inventory,
                ':timestamp': datetime.now().isoformat()
            }
        )
        
        # Calculate order total
        unit_price = float(product.get('price', 0))
        order_total = unit_price * quantity
        
        transaction_table = dynamodb.Table(transaction_name)
        transaction_id = f"ORD-{datetime.now().strftime('%Y%m%d%H%M%S')}-{str(uuid.uuid4())[:8]}"
        timestamp = datetime.now().isoformat()
        
        transaction = {
            'transaction_id': str(transaction_id),
            'timestamp': timestamp,
            'created_at': timestamp,
            'customer_info': {
                'address': '123 Demo Street',
                'phone': '(555) 123-4567',
                'city': 'San Francisco',
                'last_name': 'Doe',
                'state': 'CA',
                'first_name': 'John',
                'email': 'customer@demo.com',
                'zip_code': '94102'
            },
            'items': [{
                'name': str(product.get('name')),
                'total': Decimal(str(order_total)),
                'quantity': str(quantity),
                'price': Decimal(str(unit_price)),
                'product_id': str(product_id)
            }],
            'status': 'processing',
            'total_amount': Decimal(str(order_total)),
            'transaction_date': timestamp
        }
        
        try:
            transaction_table.put_item(Item=transaction)
        except Exception as e:
            print(f"Failed to post transaction: {str(e)}")
            raise
        
        return json.dumps({
            'status': 'order_placed',
            'product_id': product_id,
            'product_name': product.get('name'),
            'quantity_ordered': quantity,
            'unit_price': unit_price,
            'order_total': order_total,
            'previous_inventory': current_inventory,
            'new_inventory': new_inventory,
            'order_timestamp': datetime.now().isoformat()
        }, indent=2)
        
    except Exception as e:
        return json.dumps({
            'status': 'error',
            'product_id': product_id,
            'error': str(e)
        })

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### What This Code Does

* **FastMCP**: Creates an MCP server that can host your tools
* **@mcp.tool()**: Decorator that turns your Python functions into MCP tools
* **stateless_http=True**: Required for AgentCore Runtime compatibility
* **Tools**: Created 5 different tools to interact with the Dynamo DB table and Bedrock Knowledge Base

## Creating Local Testing Client

Before deploying to AgentCore Runtime, let's create a client to test our MCP server locally:

In [ ]:
%%writefile my_mcp_client.py
import asyncio
from datetime import timedelta

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    mcp_url = "http://localhost:8000/mcp"
    headers = {}

    async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_result = await session.list_tools()
            print("Available tools:")
            for tool in tool_result.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())

### Testing Locally

We can now test our MCP server locally. To do this we will need to open two different terminal sessions in our notebook. We need two terminals so one can run the MCP server and the other can run the MCP client. To open a terminal you can click "File" -> "New" -> "Terminal" in the left hand corner of the screen. 

<div style="text-align:left; margin-top: 10px;">
    <img src="images/terminal.png" width="50%"/>
</div>

We are going to need two terminals to test the MCP server.

<div style="text-align:left; margin-top: 10px;">
    <img src="images/two_terminals.png" width="50%"/>
</div>

We are going to use the first terminal to run the MCP server. First, we need to make sure we are in the correct directory. Run the following command to do so in Terminal 1.

   ```bash
   cd /home/sagemaker-user/mam407-workshop/module_3
   ```
Next we need to run the MCP server, so run the following command.

   ```bash 
   python3 mcp_server.py
   ```

Now that the MCP server is running we want to run the client which can connect to the MCP server. To do that open Terminal 2. The first step is to make sure we are in the correct directory, so run the the folling:

   ```bash
   cd /home/sagemaker-user/mam407-workshop/module_3
   ```
Next we need to run the MCP client so run the following command.

   ```bash 
   python3 my_mcp_client.py
   ```

You should see your tools listed in the output of Terminal 2, this shows the MCP client connecting to and invoking the MCP server. You can close both terminals and return to the notebook file.

## Setting up Amazon Cognito for Authentication

AgentCore Runtime requires authentication. We'll use Amazon Cognito to provide JWT tokens for accessing our deployed MCP server.

In [ ]:
import sys
import os

sys.path.append(os.path.dirname(os.getcwd()))

from utils import setup_cognito_user_pool, create_agentcore_role

In [ ]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
if cognito_config is not None:
    print("Cognito setup completed ✓")
    print(f"User Pool ID: {cognito_config.get('user_pool_id', 'N/A')}")
    print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

## Configuring AgentCore Runtime Deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch. Before we do this we will create a basic AgentCore execution role.

During the configure step, your docker file will be generated based on your application code.

<div style="text-align:left">
    <img src="images/configuring.png" width="60%"/>
</div>

In [ ]:
mcp_agentcore_role = create_agentcore_role('mcp')
mcp_role = mcp_agentcore_role['Role']['RoleName']

**Note that you will get a yellow warning message from the following cell while configuring the AgentCore Runtime. These can be ignored.** The first message will say you do not have a container engine found. We will not need a container engine locally because when we launch to AgentCore Runtime it will build our container using AWS CodeBuild. You will also see a message saying "Platform mismatch". That is because the jupyter notebook in Amazon SageMaker AI runs on amd64 and AgentCore requires ARM. However, when AWS CodeBuild builds the image it will build it on ARM making it compatible to run on AgentCore.

In [ ]:
import time
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime

boto_session = Session()
region = boto_session.region_name
print(f"Using AWS region: {region}")

required_files = ['mcp_server.py', 'requirements.txt']
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

agentcore_runtime = Runtime()

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            cognito_config['client_id']
        ],
        "discoveryUrl": cognito_config['discovery_url'],
    }
}

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=False,
    execution_role=mcp_role,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    authorizer_configuration=auth_config,
    protocol="MCP",
    agent_name="mcp_server_agentcore"
)
print("Configuration completed ✓")

## Launching MCP Server to AgentCore Runtime

Now that we've got a docker file, let's launch the MCP server to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime. One of the first lines of output will print the execution role that was created for the AgentCore Runtime. Note this as we will need to add permissions to the role so it can access Bedrock and Dynamo DB. 

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

## Checking AgentCore Runtime Status

Now that we've deployed the AgentCore Runtime, let's check for its deployment status and wait for it to be ready:

In [ ]:
print("Checking AgentCore Runtime status...")
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
print(f"Initial status: {status}")

end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    print(f"Status: {status} - waiting...")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']

if status == 'READY':
    print("✓ AgentCore Runtime is READY!")
else:
    print(f"⚠ AgentCore Runtime status: {status}")
    
print(f"Final status: {status}")

## Permissions for the Execution Role

Now we need to add additional permissions to the execution role for the agent. We will take the following IAM policy and add it as an in-line policy to the execution role for the AgentCore Runtime execution role. If you do not do this the MCP Server will have errors when invoking tools that use Dynamo DB and Bedrock. 

In [ ]:
sts = session.client('sts')
account_id = sts.get_caller_identity()['Account']

In [ ]:
import json 
from mcp_server import table_name, transaction_name, knowledge_base_id

iam_policy = {
    "Version": "2012-10-17",
	"Statement": [
		{
			"Effect": "Allow",
			"Action": [
				"dynamodb:GetItem",
				"dynamodb:PutItem",
				"dynamodb:UpdateItem",
				"dynamodb:DeleteItem",
				"dynamodb:Query",
				"dynamodb:Scan",
				"dynamodb:BatchGetItem",
				"dynamodb:BatchWriteItem"
			],
			"Resource": [
				f"arn:aws:dynamodb:{region}:{account_id}:table/{table_name}", 
				f"arn:aws:dynamodb:{region}:{account_id}:table/{transaction_name}" 
			]
		},
		{
			"Effect": "Allow",
			"Action": [
				"bedrock:Retrieve",
				"bedrock:RetrieveAndGenerate"
			],
			"Resource": f"arn:aws:bedrock:{region}:{account_id}:knowledge-base/{knowledge_base_id}"
		}
	]
}

iam_policy = json.dumps(iam_policy, indent=2)

Now that we have the proper IAM policy to allow access to the AWS resources we will update the execution role with the policy in the following code block. 

In [ ]:
import boto3

iam = boto3.client('iam')

iam.put_role_policy(
    RoleName = mcp_role,
    PolicyName = 'mcp_resource_policy',
    PolicyDocument = iam_policy
)

## Storing Configuration for Remote Access

Before we can invoke our deployed MCP server, let's store the Agent ARN and Cognito configuration in AWS Systems Manager Parameter Store and AWS Secrets Manager for easy retrieval:

In [ ]:
import boto3
import json

ssm_client = boto3.client('ssm', region_name=region)
secrets_client = boto3.client('secretsmanager', region_name=region)

try:
    cognito_credentials_response = secrets_client.create_secret(
        Name='mcp_server/cognito/credentials',
        Description='Cognito credentials for MCP server',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Cognito credentials stored in Secrets Manager")
except secrets_client.exceptions.ResourceExistsException:
    secrets_client.update_secret(
        SecretId='mcp_server/cognito/credentials',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Cognito credentials updated in Secrets Manager")

agent_arn_response = ssm_client.put_parameter(
    Name='/mcp_server/runtime/agent_arn',
    Value=launch_result.agent_arn,
    Type='String',
    Description='Agent ARN for MCP server',
    Overwrite=True
)
print("✓ Agent ARN stored in Parameter Store")

print("\nConfiguration stored successfully!")
print(f"Agent ARN: {launch_result.agent_arn}")

## Creating Remote Testing Client

Now let's create a client to test our deployed MCP server. This client will retrieve the necessary credentials from AWS and connect to the deployed server. It will be used to show the different tools the MCP Server has access to:

<div style="text-align:left">
    <img src="images/invoke.png" width="85%"/>
</div>

In [ ]:
%%writefile my_mcp_client_remote.py
import asyncio
import boto3
import json
import sys
from boto3.session import Session
from datetime import timedelta

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        print("✓ Retrieved bearer token from Secrets Manager")
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    if not agent_arn or not bearer_token:
        print("Error: AGENT_ARN or BEARER_TOKEN not retrieved properly")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")
    print("Headers configured ✓")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, 'inputSchema') and tool.inputSchema:
                        properties = tool.inputSchema.get('properties', {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()
                
                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

## Testing Your Deployed MCP Server

Let's test our deployed MCP server using the remote client:

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python my_mcp_client_remote.py

## Invoking MCP Tools Remotely

Now let's create an enhanced client that not only lists tools but also invokes them to demonstrate the full MCP functionality.


In [ ]:
%%writefile invoke_mcp_tools.py
import asyncio
import boto3
import json
import sys
from boto3.session import Session
from datetime import timedelta

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        print("✓ Retrieved bearer token from Secrets Manager")
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}: {tool.description}")
                
                print("\n🧪 Testing MCP Tools:")
                print("=" * 50)
                
                try:
                    print("\n➕ Testing kb")
                    add_result = await session.call_tool(
                        name="search_product_knowledge",
                        arguments={"query": "tell me about the running shoes"}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n➕ Testing dynamodb - check_inventory")
                    add_result = await session.call_tool(
                        name="check_inventory",
                        arguments={"product_id": "SPRT-001"}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n➕ Testing dynamodb - list_products")
                    add_result = await session.call_tool(
                        name="list_products",
                        arguments={}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n➕ Testing dynamodb - place_order")
                    add_result = await session.call_tool(
                        name="place_order",
                        arguments={"product_id": "SPRT-001", "quantity": 1}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n➕ Testing dynamodb - check_inventory post order")
                    add_result = await session.call_tool(
                        name="check_inventory",
                        arguments={"product_id": "SPRT-001"}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n➕ Testing dynamodb - get_product")
                    add_result = await session.call_tool(
                        name="get_product",
                        arguments={"product_id": "SPRT-001"}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")


                print("\n✅ MCP tool testing completed!")

                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

## Test Tool Invocation

Let's test our MCP tools by actually invoking them:

Reminder, the reason this will work and we can invoke the tools is because we added IAM permissions to the Bedrock AgentCore execution role to allow it access to the specific Dynamo DB and Bedrock resources. Also, one of the tools we will test here will actually order a product for us. So if you go back to the web UI after running the following cell, you will see the order in the "Order History" tab. 

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools.py

## 🎉 Congratulations - MCP Server Module Completed

Now that you have successfully deployed an MCP server to AgentCore Runtime, you can move on to the **next module**! We need to add an agent that can invoke and handle calling the different MCP tools. **We will do this in module 4 which is also in the jupyter notebook in the "agentcore_resources" file path.** 

---

---

---

### Reauthenticate User (Don't Recreate the User Pool)

**Only use the following code if your tokens have expired.** The bearer token used to authenticate the request to AgentCore Runtime was set up to be valid for six hours, so if this time limit expires you will need to refresh the tokens to invoke the MCP Server. This code only refreshes the MCP server token, the code to reauthenticate the strands agent is in module 4. 

### ⚠️ IMPORTANT – Manual Step Required

Copy over the client_id and region from the output earlier in the notebook when cognito was set up.

In [ ]:
from utils import reauthenticate_user
import boto3
import json

client_id = '' #paste in from the output above 
region = ''
new_bearer_token = reauthenticate_user(client_id)

secrets_client = boto3.client('secretsmanager', region_name=region)

try:
    secret_name='mcp_server/cognito/credentials'
    current_secret_response = secrets_client.get_secret_value(SecretId=secret_name)
    current_secret = json.loads(current_secret_response['SecretString'])
    
    # Update only the bearer_token field
    current_secret['bearer_token'] = new_bearer_token
    
    # Update the secret
    secrets_client.update_secret(
        SecretId=secret_name,
        SecretString=json.dumps(current_secret)
    )
    
    print(f"✓ Token refreshed and updated in Secrets Manager: {secret_name}")
    
except Exception as e:
    print(f"Error updating token in Secrets Manager: {e}")
    raise